# Notebook 2 – Machine Learning Workflow


In [1]:
import pandas as pd

df = pd.read_csv("data.csv", encoding="latin1")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


## 1. Problem Definition

What are we trying to predict, and why does it matter? **Problem:**
predict whether a transaction line will be cancelled (`IsCancelled`),
before it happens. This is a classification problem with real business
value: catching likely cancellations early supports inventory planning,
fraud review, and customer service prioritization.

## 2. Business Understanding

Understanding the domain before touching code. From the Feature
Engineering series, we already know: this is a UK-based online retailer,
cancellations are recorded as negative `Quantity` with an `InvoiceNo`
starting with "C", and the business is closed on Saturdays. This context
shapes which features are meaningful and which would be leakage (like
raw signed `Quantity`, caught directly in Notebook 1).

## 3. Data Collection

In this sprint, data collection is already done, `data.csv` is our
source. In a real project, this step would involve pulling data from a
transactional database, an e-commerce platform's API, or a data
warehouse.

## 4. Data Understanding

Basic inspection: shape, types, missing values, before any changes.

In [2]:
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())
print("\nData types:")
print(df.dtypes)

Shape: (541909, 8)

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Data types:
InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID     float64
Country            str
dtype: object


## 5. EDA (Exploratory Data Analysis)

Looking for patterns before modeling. From the Feature Engineering
series, we already found real patterns worth confirming here: seasonal
climb toward the holidays, and heavy class imbalance in `IsCancelled`.

In [3]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["IsCancelled"] = df["InvoiceNo"].astype(str).str.startswith("C").astype(int)

print("Class balance:")
print(df["IsCancelled"].value_counts(normalize=True).round(3))

monthly_orders = df.groupby(df["InvoiceDate"].dt.month)["InvoiceNo"].nunique()
print("\nOrders by month:")
print(monthly_orders)

Class balance:
IsCancelled
0    0.983
1    0.017
Name: proportion, dtype: float64

Orders by month:
InvoiceDate
1     1476
2     1393
3     1983
4     1744
5     2162
6     2012
7     1927
8     1737
9     2327
10    2637
11    3462
12    3040
Name: InvoiceNo, dtype: int64


## 6. Data Preprocessing

Cleaning and preparing the raw data, dropping rows with missing
`Description`, handling missing `CustomerID` where relevant, and
correcting types, following the Data Cleaning series.

In [4]:
df = df.dropna(subset=["Description"]).copy()
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
print("Rows after dropping missing Description:", len(df))

Rows after dropping missing Description: 540455


## 7. Feature Engineering

Building the actual model inputs, drawing directly on the earlier
Feature Engineering series. Critically, we use `AbsQuantity`, not raw
`Quantity`, to avoid the leakage caught in Notebook 1.

In [5]:
df["AbsQuantity"] = df["Quantity"].abs()
df["Month"] = df["InvoiceDate"].dt.month
df["IsInternational"] = (df["Country"] != "United Kingdom").astype(int)

feature_cols = ["AbsQuantity", "UnitPrice", "Month", "IsInternational"]
df[feature_cols].head()

,AbsQuantity,UnitPrice,Month,IsInternational
0,6,2.55,12,0
1,6,3.39,12,0
2,8,2.75,12,0
3,6,3.39,12,0
4,6,3.39,12,0


## 8. Train/Test Split

Splitting before any further statistical fitting, following the leakage
discipline from the Feature Engineering series.

In [6]:
from sklearn.model_selection import train_test_split

X = df[feature_cols].fillna(0)
y = df["IsCancelled"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (432364, 4) | Test: (108091, 4)


## 9. Model Selection

Choosing an algorithm suited to the problem. For a classification problem
with a mix of numeric features and no strong linear separability
expected, a Random Forest is a reasonable starting choice, robust to
outliers, handles non-linear patterns, doesn't require heavy scaling.

In [7]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, class_weight="balanced")
print("Model selected:", model)

Model selected: RandomForestClassifier(class_weight='balanced', max_depth=8, random_state=42)


## 10. Model Training

In [ ]:
model.fit(X_train, y_train)
print("Model trained.")

## 11. Prediction

In [ ]:
predictions = model.predict(X_test)
prediction_probabilities = model.predict_proba(X_test)[:, 1]
print("Sample predictions:", predictions[:10])
print("Sample predicted probabilities:", prediction_probabilities[:10].round(3))

## 12. Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print("ROC AUC:", round(roc_auc_score(y_test, prediction_probabilities), 3))
print("\nFull report:")
print(classification_report(y_test, predictions, zero_division=0))

## 13. Hyperparameter Tuning

Trying a small grid of hyperparameter combinations to see if performance
improves over our initial guess.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [5, 8, 12]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight="balanced"),
    param_grid,
    scoring="roc_auc",
    cv=3
)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validated ROC AUC:", round(grid_search.best_score_, 3))

## 14. Model Selection (Revisited, After Tuning)

Comparing the tuned model against our original guess, using the held-out
test set, only now, after tuning is finished.

In [ ]:
best_model = grid_search.best_estimator_
tuned_predictions = best_model.predict(X_test)
tuned_probabilities = best_model.predict_proba(X_test)[:, 1]

print("Tuned model test ROC AUC:", round(roc_auc_score(y_test, tuned_probabilities), 3))
print("Original model test ROC AUC:", round(roc_auc_score(y_test, prediction_probabilities), 3))

## 15. Deployment

The final chosen model would be saved (e.g., with `joblib`) and wrapped
in a prediction service, an API endpoint, a batch scoring job, or an
integration directly into the order-processing system, so new
transactions get scored for cancellation risk in real time. Deployment
also means monitoring: if `IsCancelled` rates shift over time (e.g., a
new product line with different return patterns), the model needs
retraining, not a permanent "set and forget" deployment.

In [ ]:
import joblib

joblib.dump(best_model, "cancellation_model.pkl")
print("Model saved for deployment: cancellation_model.pkl")

## Complete Workflow Diagram

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

steps = [
    "1. Problem\nDefinition", "2. Business\nUnderstanding", "3. Data\nCollection",
    "4. Data\nUnderstanding", "5. EDA", "6. Data\nPreprocessing",
    "7. Feature\nEngineering", "8. Train/Test\nSplit", "9. Model\nSelection",
    "10. Model\nTraining", "11. Prediction", "12. Evaluation",
    "13. Hyperparameter\nTuning", "14. Final Model\nSelection", "15. Deployment"
]

fig, ax = plt.subplots(figsize=(14, 10))
cols = 3
box_w, box_h = 3.6, 1.3
gap_x, gap_y = 0.6, 0.6

positions = []
for i, step in enumerate(steps):
    row = i // cols
    col = i % cols
    x = col * (box_w + gap_x)
    y = -row * (box_h + gap_y)
    positions.append((x, y))
    box = FancyBboxPatch((x, y), box_w, box_h, boxstyle="round,pad=0.05",
                          linewidth=1.5, edgecolor="steelblue", facecolor="#eaf2fb")
    ax.add_patch(box)
    ax.text(x + box_w/2, y + box_h/2, step, ha="center", va="center", fontsize=9, weight="bold")

# Draw arrows following reading order (left-to-right, then down)
for i in range(len(steps) - 1):
    x1, y1 = positions[i]
    x2, y2 = positions[i + 1]
    row1, col1 = i // cols, i % cols
    if col1 < cols - 1:
        start = (x1 + box_w, y1 + box_h/2)
        end = (x2, y2 + box_h/2)
    else:
        start = (x1 + box_w/2, y1)
        end = (x2 + box_w/2, y2 + box_h)
    arrow = FancyArrowPatch(start, end, arrowstyle="->", mutation_scale=15, color="gray")
    ax.add_patch(arrow)

ax.set_xlim(-0.5, cols * (box_w + gap_x))
ax.set_ylim(-((len(steps)-1)//cols + 1) * (box_h + gap_y), 1)
ax.axis("off")
ax.set_title("Complete Machine Learning Workflow", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()